# A Petri net as a reactive module: bounded reachability

A **Petri net** has *places* (which hold *tokens*) and *transitions* (the events). The number
of tokens in each place is the **marking** — the state. A transition is *enabled* when its input
places have enough tokens; *firing* it moves tokens from input to output places. When several
transitions are enabled, which one fires is **non-deterministic** (no probabilities here).

This tutorial models a small net **monolithically** as one `sugar.Module` — the whole marking is
the state, and the transition that fires each step is a non-deterministic external input — and
answers a **bounded reachability** question with Z3: *can we reach a target marking within k
firings?* (`sat` = yes, with a witness firing sequence; `unsat` = not within that bound).

## The net: a bounded queue

A queueing system with two places and two transitions:

- places: **`queue`** (jobs waiting) and **`free`** (free buffer slots); capacity `C = queue + free`
- transitions: **`arrive`** (`free >= 1` → `free-1`, `queue+1`) and **`serve`** (`queue >= 1` → `queue-1`, `free+1`)

Initial marking: `queue = 0`, `free = C` (empty buffer).

In [ ]:
import numpy as np
import z3

from zrth import LIA, Int, Var, X
from zrth import sugar
from zrth.sugar import ite
from zrth import z3 as zz3

INT = Int([1, 1])
C = 3                                    # buffer capacity

## The net as a `sugar.Module`

The marking `(queue, free)` is the module's state (`ctrl`). The transition that fires is a
non-deterministic **external** input `t` (`extl`): `0 = arrive`, `1 = serve`. Each transition is
a *guarded* update — if the chosen transition isn't enabled, the marking simply stays (a stutter).

In [ ]:
queue, free, t = Var(INT), Var(INT), Var(INT)


class BoundedQueue(sugar.Module):
    def init(self, fire):
        return 0, C                                  # empty queue, all slots free

    def update(self, q, f, fire):                    # fire: 0 = arrive, 1 = serve
        arrive = (fire == 0) & (f >= 1)
        serve = (fire == 1) & (q >= 1)
        return (ite(arrive, q + 1, ite(serve, q - 1, q)),
                ite(arrive, f - 1, ite(serve, f + 1, f)))


program = BoundedQueue(theory=LIA, ctrl=(queue, free), extl=(t,))
print(program.with_varnames({queue: "queue", free: "free", t: "t"}))
print("open (t is external)?", program.open())

module
  external
    t : Int([1,1])
  interface
    queue : Int([1,1])
    free : Int([1,1])
  atom controls queue, free reads queue, free, t
  init
    #9 := [[0]] 
    X(queue) := Id #9
    #10 := [[3]] 
    X(free) := Id #10
  delay
    d(queue) := ZERO 
    d(free) := ZERO 
  update
    #11 := [[0]] 
    #12 := Eq (t, #11)
    #13 := [[1]] 
    #14 := Ge (free, #13)
    #15 := And (#12, #14)
    #16 := [[1]] 
    #17 := Eq (t, #16)
    #18 := [[1]] 
    #19 := Ge (queue, #18)
    #20 := And (#17, #19)
    #21 := [[1]] 
    #22 := Add (queue, #21)
    #23 := [[1]] 
    #24 := Sub (queue, #23)
    #25 := Ite (#20, #24, queue)
    #26 := Ite (#15, #22, #25)
    #27 := [[1]] 
    #28 := Sub (free, #27)
    #29 := [[1]] 
    #30 := Add (free, #29)
    #31 := Ite (#20, #30, free)
    #32 := Ite (#15, #28, #31)
    X(queue) := Id #26
    X(free) := Id #32

open (t is external)? True


## Bounded reachability

Unroll the transition `steps` times: fresh markings `queue_0..queue_k`, `free_0..free_k`, and a
fresh firing choice `fire_0..fire_{k-1}` at each step. We pin the initial marking, chain each step
through the module's own `update` terms (evaluated symbolically with `zz3`), and ask whether any
step's `queue` hits the target. `sat` returns a concrete firing sequence as the witness.

In [ ]:
def reach_queue(target, steps):
    "Firing sequence reaching queue == target within `steps` steps, or None if unreachable."
    q = [z3.Int(f"queue_{i}") for i in range(steps + 1)]
    f = [z3.Int(f"free_{i}") for i in range(steps + 1)]
    fire = [z3.Int(f"fire_{i}") for i in range(steps)]

    s = z3.Solver()
    s.add(q[0] == 0, f[0] == C)                       # initial marking
    for i in range(steps):
        s.add(fire[i] >= 0, fire[i] <= 1)             # a valid transition choice
        st = {queue: np.array([[q[i]]]), free: np.array([[f[i]]]), t: np.array([[fire[i]]])}
        for atom in program.atoms:                    # apply the module's transition
            for term in atom.update:
                st.update(zip(term.write, zz3.eval(term.itype, [st[w] for w in term.read])))
        s.add(q[i + 1] == st[X(queue)].item(), f[i + 1] == st[X(free)].item())

    s.add(z3.Or(*[q[i] == target for i in range(steps + 1)]))
    if s.check() != z3.sat:
        return None
    m = s.model()
    names = ["arrive", "serve"]
    return [(names[m.eval(fire[i]).as_long()],
             (m.eval(q[i + 1]).as_long(), m.eval(f[i + 1]).as_long())) for i in range(steps)]

## Reachability questions

Two markings inside the capacity are reachable (with a witness); one beyond the capacity is not —
the `queue + free = C` invariant is enforced by the `free >= 1` guard on `arrive`, so bounded
reachability proves it can't be exceeded.

In [ ]:
for target, steps in [(2, 5), (C, 5), (C + 1, 8)]:
    witness = reach_queue(target, steps)
    print(f"queue == {target} within {steps} steps? ", "REACHABLE" if witness else "not reachable")
    if witness:
        for fired, (q, f) in witness:
            print(f"    fire {fired:6}  ->  (queue={q}, free={f})")
    print()

queue == 2 within 5 steps?  REACHABLE
    fire arrive  ->  (queue=1, free=2)
    fire arrive  ->  (queue=2, free=1)
    fire serve   ->  (queue=1, free=2)
    fire serve   ->  (queue=0, free=3)
    fire serve   ->  (queue=0, free=3)

queue == 3 within 5 steps?  REACHABLE
    fire serve   ->  (queue=0, free=3)
    fire arrive  ->  (queue=1, free=2)
    fire arrive  ->  (queue=2, free=1)
    fire arrive  ->  (queue=3, free=0)
    fire serve   ->  (queue=2, free=1)

queue == 4 within 8 steps?  not reachable



## Result

The net is modelled as a single `sugar.Module` — marking as state, the firing choice as a
non-deterministic external input — and bounded reachability is a straight unroll-and-solve with
Z3: reachable markings come back with a witness firing sequence, and the over-capacity marking is
provably unreachable within the bound.

This is the **monolithic** style, which needs no events. Modelling each place and transition as a
*separate* atom that synchronises via events (produce/consume) is the compositional next step, and
the path toward stochastic Petri nets.